# Hyperspectral Object Detection Challenge 2026
## HS-SAFD: Hyperspectral Spectral-Attention Fusion Detector
---
**Strictly Rules-Compliant Implementation:**
- **Single Detection Model**: End-to-end 16-band HS-SAFD (no multi-model ensemble)
- **Allowed Augmentation**: Single-model multi-scale + flip Test-Time Augmentation (TTA)
- **Loss**: CIoU Loss + Multi-label Focal Loss + Objectness BCE
- **Target Metric**: mAP@[0.50:0.95] across 18 authentic/counterfeit material classes


In [ ]:
# 1. Environment & GPU Verification
import os, sys, torch
print('Python Version:', sys.version)
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Model:', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'Total VRAM: {vram:.2f} GB')


In [ ]:
# 2. Clone Repository and Install Dependencies
import os
if not os.path.exists('hyperspectral-object-detection-2026') and not os.path.exists('scripts'):
    !git clone https://github.com/Gokulraj-max/hyperspectral-object-detection-2026.git
    %cd hyperspectral-object-detection-2026
elif os.path.exists('hyperspectral-object-detection-2026'):
    %cd hyperspectral-object-detection-2026
    !git pull origin main

!pip install -q tifffile pyyaml opencv-python-headless albumentations


In [ ]:
# 3. Discover Dataset Layout in /kaggle/input
from scripts.kaggle_pipeline import auto_detect_kaggle_paths, print_detection_summary
paths = auto_detect_kaggle_paths()
print_detection_summary(paths)


In [ ]:
# 4. Preview Sample Hyperspectral Cube & Pseudo-RGB
import glob, os, tifffile
import numpy as np
import matplotlib.pyplot as plt
from datasets.visualization import render_pseudo_rgb

train_dir = paths.get('train_dir') or 'data/raw/train'
cube_files = glob.glob(os.path.join(train_dir, '**', '*.npy'), recursive=True) or glob.glob(os.path.join(train_dir, '**', '*.tif'), recursive=True)
if cube_files:
    fp = cube_files[0]
    cube = np.load(fp) if fp.endswith('.npy') else tifffile.imread(fp)
    print(f'Cube file: {fp}')
    print(f'Shape: {cube.shape}, Min: {cube.min():.3f}, Max: {cube.max():.3f}')
    pseudo_rgb = render_pseudo_rgb(cube)
    plt.figure(figsize=(6, 6))
    plt.imshow(pseudo_rgb)
    plt.title(f'Pseudo-RGB Composite: {os.path.basename(fp)}')
    plt.axis('off')
    plt.show()


In [ ]:
# 5. Run Full End-to-End HS-SAFD Training & Test Inference
# Pipeline executes:
#   - Per-band normalization calculation
#   - 80/20 train/validation stratified split
#   - Single-model HS-SAFD training (Mixed Precision AMP)
#   - Test-Time Augmentation (TTA) inference across test images
#   - Automatic formatting into /kaggle/working/submission.csv with integer image_id
#   - 12-point submission rule compliance validation
!python scripts/kaggle_pipeline.py --epochs 30 --batch-size 8 --tta


In [ ]:
# 6. Verify and Inspect submission.csv
import os, pandas as pd
sub_path = '/kaggle/working/submission.csv' if os.path.exists('/kaggle/working/submission.csv') else 'outputs/submissions/submission.csv'
df = pd.read_csv(sub_path)
print(f'Total predicted bounding boxes: {len(df)}')
print(f'Unique test images: {df["image_id"].nunique()}')
print('\nColumn Data Types:')
print(df.dtypes)
print('\nClass distribution:')
print(df['class_id'].value_counts().sort_index())
print('\nFirst 10 predictions:')
display(df.head(10))
